# Provider Walkthrough

Manual, step-by-step experimentation with the llm-litmus backend against your local Ollama instance. Run cells top to bottom, or jump into a section once **Setup** has run.

Sections 1-5 cover the provider layer (Phase 1): checking Ollama is reachable, a single generation call, the tokens/sec speed test, and the two boundary-condition errors. Sections 6-8 cover the Test Registry and two-pass run loop (Phase 2): listing registered tests and running both the deterministic (MMLU) and judge-scored (TruthfulQA) pipelines end to end.

## 1. Setup

In [ ]:
from app.core.config import settings
from app.core.providers import (
    LocalOllamaProvider,
    OllamaModelNotFoundError,
    OllamaUnreachableError,
)
from app.database import get_connection, create_run, get_run, get_responses_for_run, init_db
from app.evaluations.run_loop import run_test
from app.registry import TESTS

MODEL = settings.DEFAULT_MODEL  # change to a model you have pulled locally
JUDGE_MODEL = "deepseek-r1:1.5b"  # change to a judge model you have pulled locally
provider = LocalOllamaProvider()

# Make sure the runs/responses/scores tables exist before sections 7-8 use them.
with get_connection(settings.DB_PATH) as _conn:
    init_db(_conn)

## 2. Check Ollama is reachable

In [ ]:
import httpx

print("OLLAMA_BASE_URL:", settings.OLLAMA_BASE_URL)
response = httpx.get(f"{settings.OLLAMA_BASE_URL}/api/tags", timeout=5.0)
response.raise_for_status()
pulled_models = [m["name"] for m in response.json().get("models", [])]
print("Locally pulled models:", pulled_models)

## 3. Generate a response

In [ ]:
PROMPT = "Write a two-sentence summary of why the sky is blue."

result = await provider.generate_response(MODEL, PROMPT)
print(result.text)
print()
print("prompt_tokens:", result.prompt_tokens, "| completion_tokens:", result.completion_tokens)
print("total_duration_ns:", result.total_duration_ns, "| eval_duration_ns:", result.eval_duration_ns)

## 4. Speed test

In [ ]:
speed = await provider.test_speed(MODEL)
print(f"{speed.tokens_per_second:.2f} tokens/sec "
      f"({speed.completion_tokens} tokens in {speed.eval_duration_ns / 1e9:.2f}s)")

## 5. Error handling

The two boundary conditions `LocalOllamaProvider` raises explicit exceptions for.

### Ollama unreachable

In [ ]:
unreachable_provider = LocalOllamaProvider(base_url="http://localhost:9999")
try:
    await unreachable_provider.generate_response(MODEL, "hi")
except OllamaUnreachableError as e:
    print("Got expected OllamaUnreachableError:", e)

### Model not found

In [ ]:
try:
    await provider.generate_response("definitely-not-a-real-model:latest", "hi")
except OllamaModelNotFoundError as e:
    print("Got expected OllamaModelNotFoundError:", e)

## 6. Registry — available tests

The Test Registry (`app.registry.TESTS`) is type-blind data: dataset source, prompt template, scoring type, and the callables that turn a raw dataset row into a prompt and a gold reference.

In [ ]:
for test_id, test in TESTS.items():
    print(f"{test_id}:")
    print(f"  dataset: {test.dataset_source} / {test.dataset_config} / {test.dataset_split}")
    print(f"  scoring_type: {test.scoring_type} (scorer: {test.scorer_name})")
    print(f"  prompt_template: {test.prompt_template}")
    print()

## 7. Run the full pipeline: MMLU (deterministic)

Two-pass: `run_test` fetches items, generates a response per item (Phase A), then scores every stored response (Phase B). Deterministic scoring needs no judge model, so `judge_model` is omitted.

In [ ]:
with get_connection(settings.DB_PATH) as conn:
    mmlu_run_id = create_run(
        conn,
        test_id="mmlu_elementary_math",
        target_model=MODEL,
        prompt_template_version=TESTS["mmlu_elementary_math"].prompt_template,
        sample_size=5,
    )

await run_test(
    "mmlu_elementary_math",
    target_model=MODEL,
    sample_size=5,
    provider=provider,
    db_path=settings.DB_PATH,
    run_id=mmlu_run_id,
)

with get_connection(settings.DB_PATH) as conn:
    mmlu_run = get_run(conn, mmlu_run_id)
    mmlu_responses = get_responses_for_run(conn, mmlu_run_id)
    mmlu_scores = conn.execute(
        "SELECT scores.*, responses.item_index FROM scores "
        "JOIN responses ON scores.response_id = responses.id "
        "WHERE responses.run_id = ? ORDER BY responses.item_index",
        (mmlu_run_id,),
    ).fetchall()

print("status:", mmlu_run["status"], "| error:", mmlu_run["error"])
print("target_quantization:", mmlu_run["target_quantization"])
print()
for response, score in zip(mmlu_responses, mmlu_scores):
    print(f"[{response['item_index']}] response={response['response_text']!r}")
    print(f"    score={score['score']} passed={bool(score['passed'])} rationale={score['rationale']!r}")

## 8. Run the full pipeline: TruthfulQA (judge-scored)

Same two-pass flow, but scoring calls `JUDGE_MODEL` with a grounded rubric (the dataset's own `correct_answers`/`incorrect_answers`) instead of running pure code. Judge scores are model opinion, not ground truth -- see `docs/PROJECT_DOCUMENTATION.md` on measurement honesty.

In [ ]:
with get_connection(settings.DB_PATH) as conn:
    tqa_run_id = create_run(
        conn,
        test_id="truthfulqa_generation",
        target_model=MODEL,
        judge_model=JUDGE_MODEL,
        prompt_template_version=TESTS["truthfulqa_generation"].prompt_template,
        sample_size=5,
    )

await run_test(
    "truthfulqa_generation",
    target_model=MODEL,
    judge_model=JUDGE_MODEL,
    sample_size=5,
    provider=provider,
    db_path=settings.DB_PATH,
    run_id=tqa_run_id,
)

with get_connection(settings.DB_PATH) as conn:
    tqa_run = get_run(conn, tqa_run_id)
    tqa_responses = get_responses_for_run(conn, tqa_run_id)
    tqa_scores = conn.execute(
        "SELECT scores.*, responses.item_index FROM scores "
        "JOIN responses ON scores.response_id = responses.id "
        "WHERE responses.run_id = ? ORDER BY responses.item_index",
        (tqa_run_id,),
    ).fetchall()

print("status:", tqa_run["status"], "| error:", tqa_run["error"])
print("target_quantization:", tqa_run["target_quantization"], "| judge_quantization:", tqa_run["judge_quantization"])
print()
for response, score in zip(tqa_responses, tqa_scores):
    print(f"[{response['item_index']}] prompt={response['prompt']!r}")
    print(f"    response={response['response_text'][:100]!r}")
    print(f"    score={score['score']} passed={bool(score['passed'])} rationale={score['rationale']!r}")